# Linha 2 — Performance de Mercado de Ações & Séries Temporais (B3)
## Caderno de Auditoria de ETL, Saneamento e Validação Matemática das Métricas
**Projeto Avaliativo Final de BI — Módulo 2**  
**Responsável Técnica:** Andressa Alves de Souza  
**Orientador:** Prof. Sávio Ricardo Garcia  

---
### Contextualização e Governança da Engenharia de Dados:
Este caderno experimental integra as entregas da branch `feature/01-data-pipeline` e documenta a auditabilidade do pipeline de ponta a ponta. A abordagem implementada adota a **Arquitetura Medalhão**, segregando o ciclo de vida dos dados em três camadas:
1. **Camada Bronze (Raw):** Ingestão bruta das cotações diárias dos 12 ativos da B3 via biblioteca `yfinance` para o triênio 2021-2023.
2. **Camada Prata (Silver):** Diagnóstico de qualidade, descarte de duplicidades, padronização temporal para `datetime`, saneamento de valores nulos e conversão explícita para ponto flutuante de 64 bits (`np.float64`).
3. **Camada Ouro (Gold):** Enriquecimento setorial e cálculo vetorizado das métricas de performance e risco de mercado (`ret_diario`, `base_100`, `sma_21`, `sma_50`, `peak` e `drawdown`).

Todos os caminhos operacionais são estritamente relativos via biblioteca `pathlib.Path`, assegurando conformidade contra caminhos absolutos locais.

### 0. Mapeamento Resiliente do Diretório Raiz
**Objetivo Técnico:** Configurar os apontamentos de diretórios utilizando exclusivamente caminhos relativos resilientes via `pathlib.Path`. Isso garante que o código execute de forma idêntica em qualquer sistema operacional (Windows, Linux, macOS) e no ambiente de produção do Streamlit Community Cloud.

In [3]:
from pathlib import Path
import numpy as np
import pandas as pd
import yfinance as yf

# Identificação dinâmica e resiliente do diretório raiz
NOTEBOOK_DIR = Path.cwd()
BASE_DIR = NOTEBOOK_DIR.parent if NOTEBOOK_DIR.name == "notebooks" else NOTEBOOK_DIR
DATA_RAW_DIR = BASE_DIR / "data" / "raw"
DATA_PROCESSED_DIR = BASE_DIR / "data" / "processed"

print(f"Raiz do Projeto: {BASE_DIR}")
print(f"Diretório Raw (Bronze): {DATA_RAW_DIR}")
print(f"Diretório Processed (Ouro): {DATA_PROCESSED_DIR}")

Raiz do Projeto: c:\Projeto\b3-market-performance-bi
Diretório Raw (Bronze): c:\Projeto\b3-market-performance-bi\data\raw
Diretório Processed (Ouro): c:\Projeto\b3-market-performance-bi\data\processed


### 1. Ingestão da Base Bruta da B3 via Yahoo Finance (Camada Bronze)
**Objetivo Financeiro & Técnico:** Extrair as séries históricas diárias oficiais de 12 companhias de alta liquidez distribuídas nos 4 setores estratégicos da B3 para o triênio obrigatório (01/01/2021 a 31/12/2023). O ticker da Embraer é formalizado como `EMBJ3.SA`. A chamada via `yfinance.download` retorna uma estrutura colunar hierárquica (MultiIndex), a qual é desempilhada para o formato tabular longo (*Long Format*) para manter o padrão analítico relacional.

In [4]:
# Universo de 12 Ativos Estratégicos nos 4 Setores Econômicos
TICKERS = [
    # Bens de Capital & Indústria
    "WEGE3.SA", "EMBJ3.SA",
    # Materiais Básicos & Siderurgia/Papel
    "VALE3.SA", "GGBR4.SA", "KLBN11.SA",
    # Petróleo, Gás & Energia
    "PETR4.SA", "PRIO3.SA", "EGIE3.SA",
    # Financeiro, Consumo & Locação
    "ITUB4.SA", "BBDC4.SA", "ABEV3.SA", "RENT3.SA"
]

START_DATE = "2021-01-01"
END_DATE = "2023-12-31"

print("Iniciando extração via yfinance para os 12 papéis da B3...")
df_raw = yf.download(
    tickers=TICKERS,
    start=START_DATE,
    end=END_DATE,
    interval="1d",
    group_by="column",
    auto_adjust=False,
    threads=True
)

# Desempilhamento do MultiIndex para estrutura relacional tabular (Long Format)
df_stacked = df_raw.stack(level=1, future_stack=True).reset_index()
df_stacked.rename(
    columns={
        "Date": "date",
        "Ticker": "ticker",
        "Open": "open",
        "High": "high",
        "Low": "low",
        "Close": "close",
        "Adj Close": "adj_close",
        "Volume": "volume"
    },
    inplace=True
)

print(f"Dimensões brutas extraídas: {df_stacked.shape}")
df_stacked.head()

Iniciando extração via yfinance para os 12 papéis da B3...


[*********************100%***********************]  12 of 12 completed


Dimensões brutas extraídas: (8940, 8)


Price,date,ticker,adj_close,close,high,low,open,volume
0,2021-01-04,ABEV3.SA,11.716108,15.480000,15.850000,15.380000,15.790000,17835200
1,2021-01-04,BBDC4.SA,14.483526,21.842976,22.752066,21.702478,22.752066,48196357
2,2021-01-04,EGIE3.SA,22.430525,31.128571,31.907143,30.821428,31.357141,3114020
3,2021-01-04,EMBJ3.SA,8.244199,8.370000,9.000000,8.310000,8.940000,17114300
4,2021-01-04,GGBR4.SA,13.697058,20.666666,20.666666,19.682539,19.682539,27916686


### 2. Auditoria de Qualidade da Base Bruta: Nulos e Chave Primária
**Objetivo de Engenharia:** Conduzir o diagnóstico formal de integridade sobre a base bruta (Camada Bronze). Esta auditoria averigua a presença de valores nulos (`isna().sum()`) decorrentes de eventuais feriados locais ou falhas na resposta da API, além de quantificar possíveis registros duplicados pelo par de chave natural (`ticker`, `date`).

In [5]:
print("--- DIAGNÓSTICO DE VALORES AUSENTES (CAMADA BRONZE) ---")
print(df_stacked.isna().sum())

print("\n--- DIAGNÓSTICO DE DUPLICIDADES DE PREGÃO ---")
duplicados = df_stacked.duplicated(subset=["ticker", "date"]).sum()
print(f"Registros duplicados identificados: {duplicados}")

--- DIAGNÓSTICO DE VALORES AUSENTES (CAMADA BRONZE) ---
Price
date         0
ticker       0
adj_close    0
close        0
high         0
low          0
open         0
volume       0
dtype: int64

--- DIAGNÓSTICO DE DUPLICIDADES DE PREGÃO ---
Registros duplicados identificados: 0


### 3. Saneamento, Tipagem e Classificação Setorial (Camada Prata)
**Objetivo Técnico & Negócio:** Tratar as imperfeições identificadas na auditoria bruta. O pipeline:
1. Normaliza as datas para `datetime` estrito eliminando informações residuais de fuso horário (*timezone*).
2. Remove eventuais duplicados mantendo o último registro oficial do pregão.
3. Aplica tratamento temporal de valores nulos via propagação de cotação precedente (`ffill`) seguida de recomposição retroativa (`bfill`) agrupada por papel.
4. Converte explicitamente todas as grandezas monetárias e de volume para ponto flutuante de 64 bits (`np.float64`).
5. Enriquece os dados associando a dimensão categórica do Setor Econômico oficial a cada ativo.

In [6]:
df_clean = df_stacked.copy()

# 1. Padronização e ordenação temporal
df_clean["date"] = pd.to_datetime(df_clean["date"]).dt.tz_localize(None)
df_clean = df_clean.sort_values(by=["ticker", "date"]).reset_index(drop=True)

# 2. Descarte de duplicados
df_clean = df_clean.drop_duplicates(subset=["ticker", "date"], keep="last")

# 3. Tratamento temporal contínuo de nulos (ffill / bfill) por ticker
cols_monetarias = ["open", "high", "low", "adj_close", "volume"]
for col in cols_monetarias:
    df_clean[col] = df_clean.groupby("ticker")[col].transform(lambda s: s.ffill().bfill())

# 4. Descarte residual de segurança e conversão explícita de tipos primitivos
df_clean = df_clean.dropna(subset=["adj_close"]).reset_index(drop=True)
for col in cols_monetarias:
    df_clean[col] = df_clean[col].astype(np.float64)

# 5. Adição do Setor Econômico
MAPA_SETORES = {
    "WEGE3.SA": "Bens de Capital & Indústria", "EMBJ3.SA": "Bens de Capital & Indústria",
    "VALE3.SA": "Materiais Básicos & Siderurgia/Papel", "GGBR4.SA": "Materiais Básicos & Siderurgia/Papel",
    "KLBN11.SA": "Materiais Básicos & Siderurgia/Papel", "PETR4.SA": "Petróleo, Gás & Energia",
    "PRIO3.SA": "Petróleo, Gás & Energia", "EGIE3.SA": "Petróleo, Gás & Energia",
    "ITUB4.SA": "Financeiro, Consumo & Locação", "BBDC4.SA": "Financeiro, Consumo & Locação",
    "ABEV3.SA": "Financeiro, Consumo & Locação", "RENT3.SA": "Financeiro, Consumo & Locação"
}
df_clean["setor"] = df_clean["ticker"].map(MAPA_SETORES)

print("Esquema de tipos de dados saneados:")
print(df_clean.dtypes)
print(f"\nDiagnóstico pós-saneamento (deve ser zero nulos):\n{df_clean.isna().sum()}")

Esquema de tipos de dados saneados:
Price
date         datetime64[ns]
ticker               object
adj_close           float64
close               float64
high                float64
low                 float64
open                float64
volume              float64
setor                object
dtype: object

Diagnóstico pós-saneamento (deve ser zero nulos):
Price
date         0
ticker       0
adj_close    0
close        0
high         0
low          0
open         0
volume       0
setor        0
dtype: int64


### 4. Engenharia de Variáveis Analíticas Derivadas (Camada Ouro)
**Objetivo Quantitativo Financeiro:** Calcular as métricas analíticas oficiais da Linha 2 de forma vetorizada com janelas móveis e acumuladas agrupadas por ativo (`ticker`):
- **Retorno Diário Percentual (`ret_diario`):** $\frac{P_t - P_{t-1}}{P_{t-1}}$, com preenchimento em `0.0` no pregão inicial de cada série.
- **Desempenho Relativo Normalizado (`base_100`):** $\left(\frac{P_t}{P_0}\right) \times 100$, fixando o primeiro pregão útil de janeiro de 2021 rigorosamente em 100.
- **Médias Móveis Simples (`sma_21` e `sma_50`):** Tendências de curto prazo (21 pregões) e médio prazo (50 pregões) calculadas com `min_periods=1` para evitar cortes de registros iniciais.
- **Pico Histórico (`peak`):** Máxima cotação ajustada alcançada até a respectiva data ($\text{cummax}$).
- **Rebaixamento Máximo (`drawdown`):** Queda percentual acumulada do fechamento ajustado em relação ao topo histórico ($\frac{P_t - \text{Peak}_t}{\text{Peak}_t}$).

In [7]:
dfs_curados = []

for ticker, group in df_clean.groupby("ticker", group_keys=False):
    grp = group.sort_values(by="date").copy()
    
    # 1. Retorno Diário Percentual
    grp["ret_diario"] = grp["adj_close"].pct_change().fillna(0.0).astype(np.float64)
    
    # 2. Desempenho Relativo Normalizado (Base 100 com marco inicial no 1º pregão de 2021)
    preco_inicial = grp["adj_close"].iloc[0]
    grp["base_100"] = ((grp["adj_close"] / preco_inicial) * 100.0).astype(np.float64)
    
    # 3. Médias Móveis Simples (SMA 21 de curto prazo e SMA 50 de médio prazo)
    grp["sma_21"] = grp["adj_close"].rolling(window=21, min_periods=1).mean().astype(np.float64)
    grp["sma_50"] = grp["adj_close"].rolling(window=50, min_periods=1).mean().astype(np.float64)
    
    # 4. Topo Histórico e Rebaixamento Máximo (Drawdown)
    grp["peak"] = grp["adj_close"].cummax().astype(np.float64)
    grp["drawdown"] = ((grp["adj_close"] - grp["peak"]) / grp["peak"]).astype(np.float64)
    
    dfs_curados.append(grp)

df_ouro = pd.concat(dfs_curados, ignore_index=True)
df_ouro = df_ouro.sort_values(by=["ticker", "date"]).reset_index(drop=True)
df_ouro.head()

Price,date,ticker,adj_close,close,high,low,open,volume,setor,ret_diario,base_100,sma_21,sma_50,peak,drawdown
0,2021-01-04,ABEV3.SA,11.716108,15.480000,15.85,15.380000,15.790000,17835200.0,"Financeiro, Consumo & Locação",0.000000,100.000000,11.716108,11.716108,11.716108,0.00000
1,2021-01-05,ABEV3.SA,11.965871,15.810000,16.02,15.230000,15.370000,37105800.0,"Financeiro, Consumo & Locação",0.021318,102.131788,11.840990,11.840990,11.965871,0.00000
2,2021-01-06,ABEV3.SA,12.208064,16.129999,16.23,15.630000,15.660000,28371600.0,"Financeiro, Consumo & Locação",0.020240,104.198969,11.963348,11.963348,12.208064,0.00000
3,2021-01-07,ABEV3.SA,12.147514,16.049999,16.24,15.910000,16.030001,32540100.0,"Financeiro, Consumo & Locação",-0.004960,103.682161,12.009389,12.009389,12.208064,-0.00496
4,2021-01-08,ABEV3.SA,12.510807,16.530001,16.75,16.030001,16.100000,30188800.0,"Financeiro, Consumo & Locação",0.029907,106.782958,12.109673,12.109673,12.510807,0.00000


### 5. Auditoria Matemática Estrita e Asserções de Regra de Negócio
**Objetivo de Validação & QA:** Submeter a base consolidada (Camada Ouro) a asserções lógicas automatizadas (`assert`). Este procedimento valida formalmente se os cálculos respeitam os axiomas de matemática financeira antes do empacotamento em produção:
1. **Âncora da Base 100:** Todos os 12 ativos devem iniciar rigorosamente em $100.00$.
2. **Fronteira Superior do Drawdown:** Nenhum rebaixamento pode ser positivo ($\text{Drawdown} \le 0$).
3. **Completude Estrutural:** A base final deve conter rigorosamente zero valores nulos em todas as colunas.

In [8]:
# Teste 1: Marco inicial de Base 100
primeiros_pregoes = df_ouro.groupby("ticker").first()["base_100"]
assert np.allclose(primeiros_pregoes.values, 100.0), "Falha Crítica: Nem todos os ativos iniciaram em 100.0!"
print("✔ Asserção 1 Aprovada: Todos os 12 ativos possuem base_100 == 100.0 no 1º pregão útil.")

# Teste 2: Consistência de Drawdown (estritamente não positivo)
assert (df_ouro["drawdown"] <= 1e-9).all(), "Falha Crítica: Identificado Drawdown com valor positivo!"
print("✔ Asserção 2 Aprovada: Todos os valores de Drawdown respeitam a restrição matemática (<= 0.0).")

# Teste 3: Zero valores ausentes em toda a matriz
total_nulos = df_ouro.isna().sum().sum()
assert total_nulos == 0, f"Falha Crítica: Foram identificados {total_nulos} valores ausentes na base final!"
print(f"✔ Asserção 3 Aprovada: Zero nulos confirmados em todas as variáveis analíticas e cadastrais.")

✔ Asserção 1 Aprovada: Todos os 12 ativos possuem base_100 == 100.0 no 1º pregão útil.
✔ Asserção 2 Aprovada: Todos os valores de Drawdown respeitam a restrição matemática (<= 0.0).
✔ Asserção 3 Aprovada: Zero nulos confirmados em todas as variáveis analíticas e cadastrais.


### 6. Resumo Executivo das Evidências da Base Curada
**Objetivo Analítico:** Consolidar o diagnóstico por ticker (total de sessões negociadas, preços mínimos e máximos ajustados, retorno acumulado consolidado no triênio e o pior rebaixamento experimentado). Esse resumo fornece as primeiras evidências numéricas que embasarão os testes de hipótese na EDA preliminar e as seções 7 e 8 do relatório executivo no `README.md`.

In [9]:
resumo_auditoria = df_ouro.groupby(["setor", "ticker"]).agg(
    total_pregoes=("date", "count"),
    preco_fecho_min=("adj_close", "min"),
    preco_fecho_max=("adj_close", "max"),
    retorno_trienio_pct=("base_100", lambda s: s.iloc[-1] - 100.0),
    pior_drawdown_pct=("drawdown", lambda s: s.min() * 100.0)
).round(2)

resumo_auditoria

total_pregoes  \
setor                                ticker                     
Bens de Capital & Indústria          EMBJ3.SA             745   
                                     WEGE3.SA             745   
Financeiro, Consumo & Locação        ABEV3.SA             745   
                                     BBDC4.SA             745   
                                     ITUB4.SA             745   
                                     RENT3.SA             745   
Materiais Básicos & Siderurgia/Papel GGBR4.SA             745   
                                     KLBN11.SA            745   
                                     VALE3.SA             745   
Petróleo, Gás & Energia              EGIE3.SA             745   
                                     PETR4.SA             745   
                                     PRIO3.SA             745   

                                                preco_fecho_min  \
setor                                ticker                       
Bens de Capital & Indústria          EMBJ3.SA              8.24   
                                     WEGE3.SA             20.59   
Financeiro, Consumo & Locação        ABEV3.SA             10.22   
                                     BBDC4.SA              9.30   
                                     ITUB4.SA             13.63   
                                     RENT3.SA             37.67   
Materiais Básicos & Siderurgia/Papel GGBR4.SA             12.08   
                                     KLBN11.SA            12.59   
                                     VALE3.SA             41.22   
Petróleo, Gás & Energia              EGIE3.SA             19.45   
                                     PETR4.SA              5.70   
                                     PRIO3.SA             13.66   

                                                preco_fecho_max  \
setor                                ticker                       
Bens de Capital & Indústria          EMBJ3.SA             25.66   
                                     WEGE3.SA             40.77   
Financeiro, Consumo & Locação        ABEV3.SA             14.91   
                                     BBDC4.SA             17.24   
                                     ITUB4.SA             24.09   
                                     RENT3.SA             62.37   
Materiais Básicos & Siderurgia/Papel GGBR4.SA             22.54   
                                     KLBN11.SA            20.74   
                                     VALE3.SA             71.03   
Petróleo, Gás & Energia              EGIE3.SA             28.12   
                                     PETR4.SA             25.74   
                                     PRIO3.SA             50.12   

                                                retorno_trienio_pct  \
setor                                ticker                           
Bens de Capital & Indústria          EMBJ3.SA                167.50   
                                     WEGE3.SA                  3.51   
Financeiro, Consumo & Locação        ABEV3.SA                  2.57   
                                     BBDC4.SA                 -6.93   
                                     ITUB4.SA                 24.02   
                                     RENT3.SA                 -0.25   
Materiais Básicos & Siderurgia/Papel GGBR4.SA                 30.79   
                                     KLBN11.SA                -0.80   
                                     VALE3.SA                 18.11   
Petróleo, Gás & Energia              EGIE3.SA                 25.35   
                                     PETR4.SA                228.69   
                                     PRIO3.SA                208.32   

                                                pior_drawdown_pct  
setor                                ticker                        
Bens de Capital & Indústria          EMBJ3.SA              -58.27  
                                     WEGE3.SA              -

# 7. Storytelling Financeiro: Diagnóstico e Comportamento dos Ativos (2021–2023)

## 7.1. Contexto Macrofinanceiro do Triénio
O triénio entre 2021 e 2023 na B3 caracterizou-se pela transição de um ciclo de liquidez pós-pandemia para um choque severo de política monetária interna, no qual a taxa Selic avançou expressivamente para conter pressões inflacionárias. Este cenário de juros reais elevados impôs compressão de múltiplos em setores dependentes de crédito e consumo, ao passo que companhias exportadoras e produtoras de commodities mantiveram forte geração de caixa alavancada pelo ciclo global de preços.

---

## 7.2. A Pirâmide Invertida dos Dados

### Nível 1: Contexto e Destaques Macro (Visão de Topo)
* **Heterogeneidade de Desempenho:** A dispersão de rentabilidade entre os 12 papéis confirma que o mercado acionário brasileiro no período recompensou teses com poder de precificação e geração de dividendos, enquanto penalizou negócios intensivos em capital e com alta alavancagem financeira.
* **Resiliência Setorial:** O setor de Petróleo, Gás & Energia (com destaque para PRIO3.SA e PETR4.SA) descolou-se da média do índice Ibovespa, beneficiado pelas cotações internacionais do barril de petróleo e disciplina na alocação de capital.

### Nível 2: Insights e Evidências Numéricas dos Ativos
* **Melhor Performance Relativa (Base 100):** Papéis ligados à extração e refino registraram retornos expressivos no acumulado do triénio, operando com cotações frequentemente acima das médias móveis de 21 e 50 dias (`sma_21` e `sma_50`), confirmando tendências consistentes de valorização.
* **Sensibilidade a Juros e Drawdown Acentuado:** Companhias de Consumo, Locação e Varejo Financeiro (como BBDC4.SA e RENT3.SA) enfrentaram períodos prolongados de rebaixamento máximo (`drawdown`), com desvalorizações superiores a 30% em relação aos seus picos históricos (`peak`), reflexo direto do encarecimento do custo da dívida e inadimplência.
* **Indústria e Bens de Capital:** WEGE3.SA manteve volatilidade contida e resiliência nas métricas de risco, funcionando como ativo estabilizador na carteira teórica, ao passo que EMBJ3.SA consolidou recuperação operacional gradual à medida que a aviação comercial global retomou frotas.

### Nível 3: Recomendações e Aplicação Prática de Decisão (Ação)
* **Gestão de Risco e Paradas de Proteção (*Stop Loss*):** O rastreamento contínuo das métricas de `drawdown` e o monitoramento do cruzamento de preços com a `sma_21` oferecem sinais operacionais objetivos para investidores institucionais rebalancearem exposições antes de perdas de capital severas.
* **Diversificação Decorrelacionada:** A presença de setores complementares (Exportação de Matérias-Primas vs. Utilidade Pública / Elétricas como EGIE3.SA) comprova ser indispensável para mitigar a volatilidade da carteira em ciclos prolongados de aperto monetário.

---

## 7.3. Rigor Metodológico e Limitações Analíticas
* **Ausência de Falsa Causalidade:** Em conformidade estrita com as diretrizes do Edital, reforça-se que retornos históricos passados e correlações estatísticas entre cotações e ciclos de juros **não implicam causalidade determinística direta**, servindo como diagnósticos empíricos e observacionais do comportamento de mercado.
* **Recorte Temporal:** A janela de 745 pregões úteis reflete um ciclo financeiro específico e extraordinário, devendo as conclusões serem interpretadas à luz das condicionantes macroeconômicas vigentes entre 2021 e 2023.